<img src="https://huggingface.co/datasets/FineEnvs/SmolDataEnvs/resolve/main/banner.png" width="100%">

# 1 · Teach a small model to do data science (SFT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adithya-s-k/FineEnvs/blob/main/04-smoldataenvs/notebooks/01_sft.ipynb)

**You will:** look at a verified agent trajectory, fine-tune a small model on 4,677 of them,
and watch it answer a question about a real table.

**Time:** ~15 minutes on a free Colab T4 with the defaults.

SFT is imitation. The model copies worked examples. It is the *warm start* — the next notebook
([02 · RL](https://colab.research.google.com/github/adithya-s-k/FineEnvs/blob/main/04-smoldataenvs/notebooks/02_rl.ipynb))
is where it learns from its own attempts.


In [ ]:
%pip install -q trl peft transformers datasets trackio


## The data

`SmolDataEnvs-sft` holds **4,677 complete agent trajectories**. Every one of them solved its task:
the agent explored a real Kaggle table with a shell tool, computed an answer, and a deterministic
grader confirmed it was right. So you are imitating work that is *known* correct, not work that
merely looks plausible.


In [ ]:
from datasets import load_dataset

sft = load_dataset("FineEnvs/SmolDataEnvs-sft", split="train")
print(sft)


Each row is a conversation in the format TRL expects — `messages` plus the `bash` tool schema in
`tools`. No preprocessing needed. Here is one, abbreviated:


In [ ]:
ex = sft[0]
print(f"{ex['n_turns']} turns · difficulty {ex['difficulty_tier']}\n")

for m in ex["messages"][:5]:
    body = m.get("content") or m.get("tool_calls")
    print(f"── {m['role']}\n{str(body)[:260]}\n")


## Fine-tune it

Three knobs worth knowing:

- **`MODEL`** — `SmolLM2-360M` finishes fast and proves the path. `Qwen/Qwen3.5-2B` is what the
  real runs use; it needs a bigger GPU.
- **`MAX_SAMPLES`** — how many trajectories to train on. Raise it once the loop works.
- **LoRA** — trains a small adapter instead of the whole model, which is why this fits on a T4.

`enable_thinking: False` matters: Qwen3.5 writes a `<think>` block by default, and everything
here — SFT, RL, eval — has to render the same template or you train one model and measure another.


In [ ]:
import torch
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"   # or "Qwen/Qwen3.5-2B" on a bigger GPU
MAX_SAMPLES = 200                                # None for all 4,677
HUB_MODEL_ID = None                              # e.g. "your-name/smoldataenvs-sft"

data = sft.select(range(MAX_SAMPLES)) if MAX_SAMPLES else sft
split = data.train_test_split(test_size=0.05, seed=42)

trainer = SFTTrainer(
    model=MODEL,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear"),
    args=SFTConfig(
        output_dir="smoldataenvs-sft",
        num_train_epochs=1,
        max_length=4096,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        gradient_checkpointing=True,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,
        chat_template_kwargs={"enable_thinking": False},
        bf16=torch.cuda.is_available(),
        push_to_hub=bool(HUB_MODEL_ID),
        hub_model_id=HUB_MODEL_ID,
    ),
)
trainer.train()


Loss going down and `mean_token_accuracy` going up means it is learning the shape of the work:
inspect the table, compute, answer.


## Try it

Give the model a task it has never seen and read what it writes.


In [ ]:
from transformers import pipeline

task = load_dataset("FineEnvs/SmolDataEnvs", split="eval")[0]
print("Q:", task["question"])
print("gold:", task["answer"], "\n")

gen = pipeline("text-generation", model=trainer.model, tokenizer=trainer.processing_class)
out = gen(
    [
        {"role": "system", "content": "You are a data analyst working in a sandbox."},
        {"role": "user", "content": task["question"]},
    ],
    max_new_tokens=256,
)
print(out[0]["generated_text"][-1]["content"][:600])


A small model after 200 examples will not be reliable — that is the point. It has learned the
*format*. Learning to be *right* is what the RL notebook does.


## Run it properly, on a GPU you do not own

The same training as a single script on [Hugging Face Jobs](https://huggingface.co/docs/huggingface_hub/guides/jobs).
Jobs need a token with write access, and the container is deleted when the job ends — so it pushes
to the Hub rather than to disk.


In [ ]:
# !pip install -q huggingface_hub
# !hf auth login

SHA = "main"   # pin a commit for a reproducible run
RAW = f"https://raw.githubusercontent.com/adithya-s-k/FineEnvs/{SHA}/04-smoldataenvs/scripts"

!hf jobs uv run --detach \
  --flavor a10g-large --timeout 3h --image huggingface/trl --secrets HF_TOKEN \
  -e MODEL=Qwen/Qwen3.5-2B \
  -e HUB_MODEL_ID=your-name/smoldataenvs-sft-2b \
  -e TRACKIO_SPACE_ID=your-name/trackio \
  {RAW}/train_sft.py

# watch it:  !hf jobs ps        logs:  !hf jobs logs <job-id>


## Next

**[02 · RL on SmolDataEnvs](https://colab.research.google.com/github/adithya-s-k/FineEnvs/blob/main/04-smoldataenvs/notebooks/02_rl.ipynb)**
— the model writes a program, a sandbox runs it against the real table, and the grader pays it for
being right. No imitation, no judge.

· [the dataset](https://huggingface.co/collections/FineEnvs/smoldataenvs)
· [the code](https://github.com/adithya-s-k/FineEnvs/tree/main/04-smoldataenvs)
